In [12]:
from math import log

In [13]:
def create_dataset():
    #年龄0-青年 1-中年 2-老年,
    #有工作0-1,
    #有自己的房子0-1,
    #信贷情况0-一般,1-好,2-非常好,
    #判断是否放贷
    data_set = [
        [0, 0, 0, 0, 'no'],         #数据集
        [0, 0, 0, 1, 'no'],
        [0, 1, 0, 1, 'yes'],
        [0, 1, 1, 0, 'yes'],
        [0, 0, 0, 0, 'no'],
        [1, 0, 0, 0, 'no'],
        [1, 0, 0, 1, 'no'],
        [1, 1, 1, 1, 'yes'],
        [1, 0, 1, 2, 'yes'],
        [1, 0, 1, 2, 'yes'],
        [2, 0, 1, 2, 'yes'],
        [2, 0, 1, 1, 'yes'],
        [2, 1, 0, 1, 'yes'],
        [2, 1, 0, 2, 'yes'],
        [2, 0, 0, 0, 'no']
    ]
    labels = ['年龄', '有工作', '有自己的房子', '信贷情况']
    return data_set,labels

### 构建决策树的方法
1. 特征选择
2. 决策树生成
3. 决策树修剪
### 特征选择
选定不同的特征可能会存在不同的结果,需要根据信息量的大小来选择节点构建树
#### 香农熵
信息的度量方式成为熵,该度量方式从前人经验得出
香农熵的计算公式为:
$$H(X)=-\sum_{i=1}^{n}p(x_i)\log_2p(x_i)$$
> n为分类的数目,熵越大,随机变量的不确定性越大
>
根据数据估计得到的熵,为经验熵,经验熵公式为
$$ H(D) = -\sum_{k=1}^{K} \frac{|D_k|}{|D|} \log_2 \frac{|D_k|}{|D|} $$
> D表示样本容量,Dk表示有k个类,每个类的样本容量为|Dk|
>


In [14]:
def calc_shannon_ent(data_set):
    data_set_size = len(data_set)
    label_count = {}
    #对是否放贷进行统计
    for feat_vector in data_set:
        current_label = feat_vector[-1]
        if current_label not in label_count.keys():
            label_count[current_label] = 0
        label_count[current_label] += 1
    shannon_ent = 0.0
    for kv in label_count:
        prob = float(label_count[kv]) / data_set_size
        shannon_ent -= prob * log(prob,2)
    return shannon_ent

### 信息增益
构建决策树的核心就是找到信息增益最大的变量作为根
信息增益越大,对最终分类影响越大,因此还需要求得信息增益
信息增益需要通过条件熵求得
条件熵计算公式
$$ H(Y|X) = \sum_{x \in X} p(x) * H(Y|X=x) $$
$$ H(Y|X) = -\sum_{x \in X} p(x) \sum_{y \in Y} p(y|x) \log_2 p(y|x) $$
同理,当熵的概率由数据估计得到的时候,所对应的条件熵为**条件经验熵**
> 当 X = x 的时候, 相当于X 取到了特定值，因此X不存在混乱特征,所以需要剔除掉X这个特征
>
信息增益中,是相对的概念,比如特征A对训练数据集D的信息增益为集合D的经验熵与特征A条件下的D的经验条件熵的差,因此信息增益的计算公式为:
$$ G(D,A) = H(D) - H(D|A) $$
> 一般的,将熵和条件熵作差为互信息,在决策树中,互信息作为信息增益的度量
>
信息增益越大,越适合作为根节点,因为对最终分类效果影响最大

In [15]:
def split_data_set(data_set, idx, value):
    res_data_set = []
    for feat_vec in data_set:
        if feat_vec[idx] == value:
            reduce_feat_vec = feat_vec[:idx]
            reduce_feat_vec.extend(feat_vec[idx+1:])
            res_data_set.append(reduce_feat_vec)
    return res_data_set

def choose_best_feature_to_split(data_set):
    num_feature = len(data_set[0]) - 1
    base_entropy = calc_shannon_ent(data_set)
    best_info_increase = 0.0
    best_feature_idx = -1
    for i in range(num_feature):
        #从data_set取出某一列
        feature_list = [example[i] for example in data_set]
        #使用set去重
        unique_val = set(feature_list)
        condition_entropy = 0.0
        for val in unique_val:
            sub_data_set = split_data_set(data_set, i ,val)
            #计算该分类出现的概率
            prob = len(sub_data_set) / float(len(data_set))
            condition_entropy += prob * calc_shannon_ent(sub_data_set)
        info_increase = base_entropy - condition_entropy
        print("第%d个特征,信息增益为%.3f" % (i, info_increase))
        if info_increase > best_info_increase:
            best_info_increase = info_increase
            best_feature_idx = i
    return best_feature_idx

### ID3算法构建决策树
1. 从根节点递归的构建决策树
2. 对节点计算所有的可能特征的信息增益,并选择信息增益最大的特征作为节点
3. 由特征的不同取值作为子节点 , 比如是否有房 (root: 是否 left: 是 right: 否)

In [16]:
def majority_cnt(class_list):
    class_count = {}
    for vote in class_list:
        if vote not in class_count.keys():
            class_count[vote] = 0
        class_count[vote] += 1
    sorted_class_count = sorted(class_count.items(), key=lambda x:x[1], reverse=True)
    return sorted_class_count[0][0]
#构建决策树
def create_tree(data_set, labels, feat_labels):
    #取是否放贷
    class_list = [example[-1] for example in data_set]
    #如果类别完全相同,即为所有的都为"是"
    if class_list.count(class_list[0]) == len(class_list):
        return class_list[0]
    #遍历完所有特征，即为最后只剩下 是否放贷这个标签
    if len(data_set[0]) == 1 or len(labels) ==0:
        return majority_cnt(class_list)
    #选择最优特征
    best_feature_idx = choose_best_feature_to_split(data_set)
    best_feature_labels = labels[best_feature_idx]
    #选择的特征列表
    feat_labels.append(best_feature_labels)
    #构建树
    my_tree = {best_feature_labels:{}}
    #删除已经选择的特征
    del(labels[best_feature_idx])
    #得到最优特征里的所有值 ，比如是否有房中的 是 or 否
    feat_values = [example[best_feature_idx] for example in data_set]
    unique_values = set(feat_values)
    for value in unique_values:
        #浅拷贝一个labels (删除完标签的)
        sub_labels = labels[:]
        my_tree[best_feature_labels][value] = create_tree(split_data_set(data_set, best_feature_idx, value), sub_labels, feat_labels)
    return my_tree

### 以下是绘制决策树 使用matplotlib
1. get_num_leaves 获取叶子节点的个数
2. get_tree_depth 获取树深度
3. plot_node 绘制节点
4. plot_mid_text 标注有向边属性值
5. plot_tree 绘制树
6. create_plot 创建面板

In [17]:
from matplotlib.font_manager import FontProperties
import matplotlib.pyplot as plt

In [18]:
def get_num_leaves(my_tree):
    num_leaves = 0
    if type(my_tree) != dict:
        return 1
    for key, value in my_tree.items():
        num_leaves += get_num_leaves(value)
    return num_leaves
def get_tree_depth(my_tree):
    max_depth = 0
    if type(my_tree) != dict:
        return 1
    for key,value in my_tree.items():
        depth = get_tree_depth(value)
        if depth > max_depth:
            max_depth = depth
        else:
            max_depth += 1
    return max_depth

In [19]:
#使用决策树进行分类，根据节点一个个判断即可
def classify(decision_tree, feat_labels, test_vec):
    #取根节点
    feat_list = list(decision_tree.keys())
    root_node = feat_list[0]
    feat_idx = feat_labels.index(root_node)
    son_tree = decision_tree.get(root_node)
    for key in son_tree.keys():
        # key 为 [ 0 , { 有工作 : {} } ]
        if test_vec[feat_idx] == key:
            #如果为 {有工作: {}} 继续递归
            if type(son_tree[key]) == dict:
                class_label = classify(son_tree[key], feat_labels, test_vec)
            else: class_label = son_tree[key]
    return class_label

In [24]:
# if __name__ == "__main__":
#     data_set, labels = create_dataset()
#     my_tree = create_tree(data_set, labels, [])
#     test_vec = [1,1]
#     feat_labels = ['有自己的房子','有工作']
#     result = classify(my_tree, feat_labels, test_vec)
#     print("测试结果为: %s" % result)

第0个特征,信息增益为0.083
第1个特征,信息增益为0.324
第2个特征,信息增益为0.420
第3个特征,信息增益为0.363
第0个特征,信息增益为0.252
第1个特征,信息增益为0.918
第2个特征,信息增益为0.474
测试结果为: yes


### 决策树存储
需要存储决策树，依次保证以后调用的可行性，否则每次都需要训练一次..

In [27]:
import pickle

def store_tree(input_tree, filename):
    with open(filename, 'wb') as fw:
        pickle.dump(input_tree, fw)

def load_tree(filename):
    fr = open(filename, 'rb')
    return pickle.load(fr)

if __name__ == '__main__':
    myTree = {'有自己的房子': {0: {'有工作': {0: 'no', 1: 'yes'}}, 1: 'yes'}}
    store_tree(myTree, 'classifierStorage.txt')
    print(load_tree('classifierStorage.txt'))

{'有自己的房子': {0: {'有工作': {0: 'no', 1: 'yes'}}, 1: 'yes'}}
